# Experiment: Recovering X-Wines Final Slide Analysis

Objective:
- Recover the core notebook so it follows the finished June 14, 2024 slide deck.
- Keep a quick sample mode for committed outputs and a full mode for deeper reruns.

Success criteria:
- The notebook matches the slide narrative from framing through recommendation ranking and the appendix models.
- Every path is repo-relative and the heavy lifting lives in reusable project code.
- Sample-mode outputs are lightweight enough to commit, while full mode remains available for closer reproduction.


In [1]:
# Setup: imports, display helpers, and repo-relative module access
from __future__ import annotations

import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from wine_rating_inference.xwines_recovery import (
    NotebookRunConfig,
    aggregate_recommendation_frame,
    build_data_overview,
    fit_recommendation_model,
    load_wine_catalog,
    prepare_red_wine_frame,
    run_decision_tree_appendix,
    run_naive_bayes_appendix,
    slide_groupby_snippet,
)

pd.options.display.max_columns = 20
pd.options.display.float_format = "{:,.6f}".format


def confusion_matrix_frame(matrix):
    return pd.DataFrame(
        matrix,
        index=["actual_0", "actual_1"],
        columns=["pred_0", "pred_1"],
    )

{
    "project_root": ".",
    "src_path": "src",
}


{'project_root': '.', 'src_path': 'src'}

## Run Mode

The recovered notebook has two execution modes:

- `sample`: read a deterministic first slice of the cleaned panel, lower the minimum `winery × vintage` threshold, and keep committed outputs light.
- `full`: use the slide-aligned `>= 500` label-frequency rule and aim for a closer reproduction of the final presentation.

The committed notebook is executed in `sample` mode on purpose. If you want to chase the slide numbers more closely, switch `RUN_MODE` to `"full"` and rerun from the top.


In [2]:
# Top-level runtime configuration
RUN_MODE = "sample"  # change to "full" for a closer reproduction of the finished slides

CONFIG = NotebookRunConfig(
    run_mode=RUN_MODE,
    top_n=15 if RUN_MODE == "sample" else 50,
    sample_nrows=300_000,
    sample_min_label_count=100,
    full_min_label_count=500,
    sample_classification_rows=10_000,
    full_classification_rows=50_000,
)

{
    "run_mode": CONFIG.run_mode,
    "panel_path": str(CONFIG.panel_path.relative_to(ROOT)),
    "sample_nrows": CONFIG.sample_nrows,
    "minimum_label_count": CONFIG.minimum_label_count,
    "top_n": CONFIG.top_n,
    "classification_row_cap": CONFIG.classification_row_cap,
}


{'run_mode': 'sample',
 'panel_path': 'data/cleaned/Xwines.csv',
 'sample_nrows': 300000,
 'minimum_label_count': 100,
 'top_n': 15,
 'classification_row_cap': 10000}

## Data Overview Tied To The Slides

The final deck introduces X-Wines as a large web-collected wine dataset with over 20 million observations and a broad country/feature footprint. The notebook keeps that framing, but in sample mode it intentionally reads only a deterministic slice of the recovered cleaned panel so the committed analysis stays runnable.


In [3]:
# Small catalog-level overview using the recovered 100K wine metadata
overview = build_data_overview(CONFIG)
catalog = load_wine_catalog(CONFIG)

display(pd.Series(overview))
display(catalog.head(5))


catalog_rows                                                            100646
candidate_cardinality_sum                                                37373
candidate_cardinalities      {'Type': 6, 'Acidity': 3, 'Grapes': 7107, 'Bod...
dtype: object

,WineID,Type,Elaborate,Grapes,ABV,Body,Acidity,Country,WineryName
0,100001,Sparkling,Varietal/100%,['Muscat/Moscato'],7.500000,Medium-bodied,High,Brazil,Casa Perini
1,100002,Red,Varietal/100%,['Ancellotta'],12.000000,Medium-bodied,Medium,Brazil,Casa Perini
2,100003,Red,Varietal/100%,['Cabernet Sauvignon'],12.000000,Full-bodied,High,Brazil,Castellamare
3,100004,White,Varietal/100%,['Muscat/Moscato'],12.000000,Medium-bodied,Medium,Brazil,Monte Paschoal
4,100005,Red,Assemblage/Bordeaux Red Blend,"['Cabernet Sauvignon', 'Merlot']",11.000000,Full-bodied,Medium,Brazil,Aurora


## Initial OLS Framing And The Dimensionality Problem

The slides begin with a familiar OLS-style idea: predict rating from observable market features such as `Type`, `Acidity`, `Grapes`, `Body`, `Country`, and `Vintage`. The problem is not the idea itself; it is the combinatorial explosion in unique categorical values.

The final deck turns this into the key methodological pivot: even before building a full dummy matrix, the feature space is already too wide to treat naively.


In [4]:
# Show why the initial feature framing becomes awkward very quickly
cardinality_frame = (
    pd.Series(overview["candidate_cardinalities"], name="unique_values")
    .sort_values(ascending=False)
    .to_frame()
)

display(cardinality_frame)
{
    "candidate_cardinality_sum": overview["candidate_cardinality_sum"],
    "slide_message": "The finished deck treats this as a curse-of-dimensionality problem before moving to winery × vintage.",
}


,unique_values
WineryName,30190
Grapes,7107
Country,62
Type,6
Body,5
Acidity,3


{'candidate_cardinality_sum': 37373,
 'slide_message': 'The finished deck treats this as a curse-of-dimensionality problem before moving to winery × vintage.'}

## Sommelier's Knowledge And The `winery × vintage` Pivot

The final slides argue that winery technique and year-specific climate carry a large share of the taste signal that the raw feature list struggles to capture cleanly. That is the reason for moving away from the original wide-feature framing and toward a single categorical factor built from `WineryID × Vintage`.


In [5]:
# Recover the red-wine analysis frame and inspect the candidate winery × vintage labels
deduped_red, eligible_red, cleaning_summary = prepare_red_wine_frame(CONFIG)
label_count_preview = (
    eligible_red["label_display"]
    .value_counts()
    .rename_axis("label_display")
    .reset_index(name="rating_count")
    .head(10)
)

display(pd.Series(cleaning_summary))
label_count_preview


loaded_rows            300000
red_rows               240532
deduped_rows           226451
eligible_rows          170464
eligible_labels           572
minimum_label_count       100
dtype: int64

,label_display,rating_count
0,Marqués de Riscal 2014,3872
1,Marqués de Riscal 2015,3831
2,San Marzano 2012,2977
3,Marqués de Riscal 2013,2962
4,Marqués de Riscal 2011,2614
5,Caymus 2015,2372
6,Caymus 2018,2334
7,Caymus 2014,2321
8,Caymus 2013,2289
9,Caymus 2016,2227


## Data Processing & Cleansing

The recovered cleaning workflow follows the final slides in this order:

1. Keep the most recent rating when the same user rated the same wine multiple times.
2. Restrict the core analysis to red wines.
3. Filter out thin `winery × vintage` cells.
4. Average ratings at the `WineID × Label` level.

That last step corresponds to the slide's explicit `groupby(['WineID', 'Label'])['Rating'].mean()` move.


In [6]:
# Aggregate to the wine-label level that the slides use in the recommendation stage
aggregated = aggregate_recommendation_frame(eligible_red)

display(aggregated.head(10))
{
    "aggregated_rows": len(aggregated),
    "eligible_labels": int(aggregated["label_key"].nunique()),
    "slide_groupby_snippet": slide_groupby_snippet(),
}


,WineID,WineName,label_key,label_display,WineryName,Vintage,Rating
0,155523,Premium 50 Barricas Syrah,36127_2012,Alceño 2012,Alceño,2012,3.815851
1,155523,Premium 50 Barricas Syrah,36127_2016,Alceño 2016,Alceño,2016,3.842282
2,135997,Barolo,41211_1996,Borgogno 1996,Borgogno,1996,3.972222
3,136140,Barolo Riserva,41211_1996,Borgogno 1996,Borgogno,1996,4.408654
4,135997,Barolo,41211_2003,Borgogno 2003,Borgogno,2003,4.583333
5,136140,Barolo Riserva,41211_2003,Borgogno 2003,Borgogno,2003,4.357143
6,135997,Barolo,41211_2004,Borgogno 2004,Borgogno,2004,4.265957
7,136140,Barolo Riserva,41211_2004,Borgogno 2004,Borgogno,2004,4.403409
8,135997,Barolo,41211_2006,Borgogno 2006,Borgogno,2006,4.305970
9,136140,Barolo Riserva,41211_2006,Borgogno 2006,Borgogno,2006,4.397196


{'aggregated_rows': 705,
 'eligible_labels': 572,
 'slide_groupby_snippet': "df = df.groupby(['WineID', 'Label'])['Rating'].mean().reset_index()"}

## Recommendation Model, Significance Screening, And Top List

Instead of rebuilding a huge dense dummy matrix inside the notebook, the recovered version computes the one-factor OLS quantities from the grouped regression structure. This keeps the methodology aligned with the slides while making the implementation smaller and easier to maintain.

For interpretability, the notebook uses the lowest-mean eligible label as the treatment baseline. Coefficients therefore read as rating uplift relative to the weakest eligible `winery × vintage` cell in the current run mode.


In [7]:
# Fit the recommendation-stage one-factor model and inspect the baseline assumptions
recommendation = fit_recommendation_model(
    aggregated,
    familywise_alpha=CONFIG.familywise_alpha,
    top_n=CONFIG.top_n,
)

display(
    pd.Series(
        {
            "baseline_display": recommendation.baseline_display,
            "baseline_mean": recommendation.baseline_mean,
            "df_resid": recommendation.df_resid,
            "residual_mse": recommendation.residual_mse,
            "bonferroni_threshold": recommendation.bonferroni_threshold,
        }
    )
)
recommendation.coefficient_table.head(15)


baseline_display        Riunite 2019
baseline_mean               2.326023
df_resid                         133
residual_mse                0.087736
bonferroni_threshold        0.003333
dtype: object

,label_key,label_display,group_size,group_mean,coefficient,standard_error,t_value,p_value,is_reference
0,29223_1982,Château Latour 1982,2,4.916176,2.590153,0.296202,8.744542,0.000000,False
1,31555_2000,Château Cheval Blanc 2000,1,4.894958,2.568935,0.362772,7.081399,0.000000,False
2,30059_1989,Pétrus 1989,1,4.885093,2.559070,0.362772,7.054206,0.000000,False
3,23606_2014,Domaine de La Romanée-Conti 2014,4,4.872435,2.546412,0.256519,9.926809,0.000000,False
4,23606_1990,Domaine de La Romanée-Conti 1990,4,4.872076,2.546053,0.256519,9.925409,0.000000,False
5,30059_1990,Pétrus 1990,1,4.868421,2.542398,0.362772,7.008249,0.000000,False
6,29845_1990,Château Haut-Brion 1990,1,4.851010,2.524987,0.362772,6.960254,0.000000,False
7,31555_1990,Château Cheval Blanc 1990,1,4.849515,2.523491,0.362772,6.956132,0.000000,False
8,29220_2003,Château Lafite Rothschild 2003,1,4.846774,2.520751,0.362772,6.948578,0.000000,False
9,63066_1998,Penfolds 1998,1,4.846535,2.520511,0.362772,6.947918,0.000000,False


In [8]:
# Recover the slide-style ranking table after Bonferroni filtering
top_recommendations = recommendation.bonferroni_table[
    ["label_display", "coefficient", "t_value", "p_value", "group_mean", "group_size"]
].rename(
    columns={
        "label_display": "Wine",
        "coefficient": "Coefficient",
        "t_value": "t-value",
        "p_value": "P-value",
        "group_mean": "Mean Rating",
        "group_size": "Wine Count",
    }
)

top_recommendations


,Wine,Coefficient,t-value,P-value,Mean Rating,Wine Count
0,Château Latour 1982,2.590153,8.744542,0.000000,4.916176,2
1,Château Cheval Blanc 2000,2.568935,7.081399,0.000000,4.894958,1
2,Pétrus 1989,2.559070,7.054206,0.000000,4.885093,1
3,Domaine de La Romanée-Conti 2014,2.546412,9.926809,0.000000,4.872435,4
4,Domaine de La Romanée-Conti 1990,2.546053,9.925409,0.000000,4.872076,4
5,Pétrus 1990,2.542398,7.008249,0.000000,4.868421,1
6,Château Haut-Brion 1990,2.524987,6.960254,0.000000,4.851010,1
7,Château Cheval Blanc 1990,2.523491,6.956132,0.000000,4.849515,1
8,Château Lafite Rothschild 2003,2.520751,6.948578,0.000000,4.846774,1
9,Penfolds 1998,2.520511,6.947918,0.000000,4.846535,1


## Winery Aspect / Interpretation

The final deck uses the recommendation-stage results to make a practical point: even strong wineries do not produce stable outcomes every year, and most winery-year cells never become reliable recommendations. That is why the notebook reports both the raw number of conventionally significant coefficients and the much smaller Bonferroni-filtered subset.


In [9]:
# Compare the raw significance count with the stricter recommendation shortlist
raw_significant_count = int(recommendation.coefficient_table["p_value"].le(0.05).fillna(False).sum())

display(
    pd.Series(
        {
            "labels_in_model": int(recommendation.label_summary["label_key"].nunique()),
            "coefficients_below_0.05": raw_significant_count,
            "bonferroni_selected": int(len(recommendation.bonferroni_table)),
        }
    )
)
recommendation.label_summary.sort_values(["group_mean", "group_size"], ascending=[False, False]).head(10)


labels_in_model            572
coefficients_below_0.05    571
bonferroni_selected         15
dtype: int64

,label_key,label_display,group_size,group_mean
571,29223_1982,Château Latour 1982,2,4.916176
570,31555_2000,Château Cheval Blanc 2000,1,4.894958
569,30059_1989,Pétrus 1989,1,4.885093
568,23606_2014,Domaine de La Romanée-Conti 2014,4,4.872435
567,23606_1990,Domaine de La Romanée-Conti 1990,4,4.872076
566,30059_1990,Pétrus 1990,1,4.868421
565,29845_1990,Château Haut-Brion 1990,1,4.851010
564,31555_1990,Château Cheval Blanc 1990,1,4.849515
563,29220_2003,Château Lafite Rothschild 2003,1,4.846774
562,63066_1998,Penfolds 1998,1,4.846535


## Appendix: Naive Bayes For Red Wine

The slide deck later switches from the recommendation story to a binary classification appendix. The recovered notebook keeps that structure separate. The target is mapped to two classes:

- `0`: `Rating <= 1`
- `1`: `Rating >= 4`

This section uses the slide features `Elaborate`, `Grapes`, `ABV`, `Body`, and `Acidity`.


In [10]:
# Rebuild the Naive Bayes appendix on a deterministic extreme-rating subset
nb_result = run_naive_bayes_appendix(
    deduped_red_frame=deduped_red,
    max_rows=CONFIG.classification_row_cap,
    random_seed=CONFIG.random_seed,
)
nb_report = pd.DataFrame(nb_result.report).T

display(
    pd.Series(
        {
            "dataset_size": nb_result.dataset_size,
            "positive_share": nb_result.positive_share,
            "trace_share": nb_result.extra["trace_share"],
            "precision_class_0": nb_result.report["0"]["precision"],
            "precision_class_1": nb_result.report["1"]["precision"],
        }
    )
)
display(confusion_matrix_frame(nb_result.confusion_matrix))
nb_report.loc[["0", "1", "accuracy", "macro avg", "weighted avg"]]


dataset_size        2,468.000000
positive_share          0.500000
trace_share             0.619125
precision_class_0       0.618893
precision_class_1       0.619355
dtype: float64

,pred_0,pred_1
actual_0,190,118
actual_1,117,192


,precision,recall,f1-score,support
0,0.618893,0.616883,0.617886,308.000000
1,0.619355,0.621359,0.620355,309.000000
accuracy,0.619125,0.619125,0.619125,0.619125
macro avg,0.619124,0.619121,0.619121,617.000000
weighted avg,0.619124,0.619125,0.619123,617.000000


## Appendix: Decision Tree

The final slides then move to a classification tree with randomized search over the same hyperparameter family shown in the deck. The recovered notebook keeps that appendix separate from the recommendation mainline so the reader can distinguish the stronger part of the project from the more exploratory extension.


In [11]:
# Rebuild the decision-tree appendix and expose the tuned hyperparameters
tree_result = run_decision_tree_appendix(
    deduped_red_frame=deduped_red,
    max_rows=CONFIG.classification_row_cap,
    random_seed=CONFIG.random_seed,
)
tree_report = pd.DataFrame(tree_result.report).T

display(
    pd.Series(
        {
            "dataset_size": tree_result.dataset_size,
            "positive_share": tree_result.positive_share,
            "best_params": str(tree_result.extra["best_params"]),
            "precision_class_0": tree_result.report["0"]["precision"],
            "precision_class_1": tree_result.report["1"]["precision"],
        }
    )
)
display(confusion_matrix_frame(tree_result.confusion_matrix))
tree_report.loc[["0", "1", "accuracy", "macro avg", "weighted avg"]]


dataset_size                                                      2468
positive_share                                                0.500000
best_params          {'classifier__criterion': 'entropy', 'classifi...
precision_class_0                                             0.651246
precision_class_1                                             0.627976
dtype: object

,pred_0,pred_1
actual_0,183,125
actual_1,98,211


,precision,recall,f1-score,support
0,0.651246,0.594156,0.621392,308.000000
1,0.627976,0.682848,0.654264,309.000000
accuracy,0.638574,0.638574,0.638574,0.638574
macro avg,0.639611,0.638502,0.637828,617.000000
weighted avg,0.639592,0.638574,0.637855,617.000000


## Conclusion And Recovery Notes

This notebook is now the canonical analysis notebook for the repo.

What changed relative to the recovered legacy file:

- Windows, Colab, and SQLite detours are gone.
- The notebook is organized to follow the final slide deck directly.
- Heavy logic is pushed into reusable project code under `src/`.
- `sample` mode exists for fast committed outputs, while `full` mode preserves a path toward closer reproduction of the original presentation.

The remaining gap is no longer notebook chaos; it is result fidelity. If we want to chase the exact slide tables later, the next step is to tune the full-mode inputs and thresholds until the ranking and appendix metrics align even more closely with the June 14, 2024 deck.


In [12]:
# Final recovery summary
{
    "canonical_notebook": "Xwines.ipynb",
    "legacy_notebook": "docs/legacy/Xwines-original.ipynb",
    "run_mode": CONFIG.run_mode,
    "full_mode_note": "Switch RUN_MODE to 'full' for the slide-aligned >= 500 label filter.",
}


{'canonical_notebook': 'Xwines.ipynb',
 'legacy_notebook': 'docs/legacy/Xwines-original.ipynb',
 'run_mode': 'sample',
 'full_mode_note': "Switch RUN_MODE to 'full' for the slide-aligned >= 500 label filter."}